## Load the sequence

In [2]:
with open("../output/combined_text.txt", "r", encoding="utf-8") as f:
    text_sequence = f.read()

len(text_sequence)

204889

## BPE algorithm

I am using the [minBPE](https://github.com/karpathy/minbpe) repository to tokenize the sequence of text.

import sys
sys.path.insert(0, "/content/personalize_chatbot")


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\abdur\\Downloads\\personalize_chatbot\\output\\combined_text.txt'

In [16]:

import os, sys

# 🔧 CHANGE THIS to your project root (folder that contains minbpe/, output/, etc.)
PROJECT_ROOT = r"C:\Users\abdur\Downloads\Train_Your_Language_Model_Course-main\personalize_chatbot"

# Make sure Python can import your local package
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Quick sanity checks
assert os.path.isdir(os.path.join(PROJECT_ROOT, "minbpe")), "minbpe/ folder not found"
assert os.path.isfile(os.path.join(PROJECT_ROOT, "minbpe", "__init__.py")), "__init__.py missing in minbpe/"

from minbpe.regex import RegexTokenizer

# Paths
training_file = os.path.join(PROJECT_ROOT, "output", "combined_text.txt")
save_dir = os.path.join(PROJECT_ROOT, "output", "tokenizer")
save_path = os.path.join(save_dir, "darija_tokenizer.model")

# Ensure output folder exists and file is present
os.makedirs(save_dir, exist_ok=True)
assert os.path.isfile(training_file), f"Training file not found: {training_file}"

# Train
with open(training_file, "r", encoding="utf-8") as f:
    text = f.read()

tok = RegexTokenizer()
tok.train(text,vocab_size=5000)

# Save
tok.save(save_path)
print("✅ Tokenizer saved at:", save_path)


100%|██████████| 4744/4744 [05:25<00:00, 14.57it/s]

✅ Tokenizer saved at: C:\Users\abdur\Downloads\Train_Your_Language_Model_Course-main\personalize_chatbot\output\tokenizer\darija_tokenizer.model


Start by training the tokenizer on the text sequence that you saved in the previous notebook.

In [11]:
from minbpe import BasicTokenizer

tokenizer = BasicTokenizer()
tokenizer.train(text_sequence, vocab_size=1024)

NameError: name 'text_sequence' is not defined

Visualize the vocabulary.

In [ ]:
vocab = tokenizer.vocab
vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

Test the tokenizer.

In [ ]:
tokenizer.encode("Happy")

[441, 121]

In [ ]:
tokenizer.decode([441, 121])

'Happy'

Add special tokens to the vocabulary. These tokens are going to be used a lot in the fine-tuning step.

In [ ]:
max_vocab_id = list(tokenizer.vocab.keys())[-1]
tokenizer.special_tokens = {
    "<|startoftext|>": max_vocab_id + 1,
    "<|separator|>": max_vocab_id + 2,
    "<|endoftext|>": max_vocab_id + 3,
    "<|unk|>": max_vocab_id + 4,
    "<|padding|>": max_vocab_id + 5,
}

I have more than 618K tokens for training and validation. This is pretty good, but if you can add more, that would be even better.

In [ ]:
len(tokenizer.encode(text_sequence))

95379

Save the tokenizer

In [ ]:
tokenizer.save(file_prefix="../output/tokenizer/my_tokenizer")